# ETL — Extracción de tweets sobre fraude en X (Twitter)

Este notebook descarga publicaciones de X relacionadas con fraude financiero usando el endpoint `search/all` de la API v2.  
**Requisitos:** acceso de nivel Enterprise / Academic / DSA al endpoint completo, y un *bearer token* configurado en `.env`.

El CSV resultante se guarda en `data/raw/`, que **no se sube a GitHub** (ver `.gitignore`).

In [ ]:
import os
import time
import requests
import pandas as pd
pd.set_option('display.max_colwidth', None)

In [ ]:
BASE_URL = "https://api.x.com/2/tweets/search/all"

In [ ]:
def search_all_posts_to_df(
    bearer_token: str,
    query: str,
    max_pages: int = 40,
    max_results: int = 500,
    sleep_seconds: float = 1.1,
    start_time: str = None,
    end_time: str = None,
) -> pd.DataFrame:
    """
    Descarga posts desde el archivo completo de X que matcheen query.
    Requiere acceso a search/all (Enterprise / Academic / DSA).
    """
    headers = {"Authorization": f"Bearer {bearer_token}"}

    params = {
        "query": query,
        "max_results": max_results,
        "tweet.fields": ",".join([
            "id", "text", "created_at", "author_id", "lang", "geo", "public_metrics"
        ]),
        "expansions": ",".join([
            "author_id",
            "geo.place_id"
        ]),
        "user.fields": ",".join([
            "id", "name", "username", "created_at", "location"
        ]),
        "place.fields": ",".join([
            "id", "full_name", "country", "country_code", "place_type", "geo"
        ]),
        "sort_order": "recency",
    }

    if start_time:
        params["start_time"] = start_time
    if end_time:
        params["end_time"] = end_time

    all_rows = []
    next_token = None
    target_rows = max_pages * max_results

    for page in range(max_pages):
        if next_token:
            params["next_token"] = next_token
        else:
            params.pop("next_token", None)

        # --- Manejo de rate limits ---
        retries = 0
        while True:
            r = requests.get(BASE_URL, headers=headers, params=params, timeout=30)

            if r.status_code == 200:
                break
            elif r.status_code == 429:
                wait = min(60 * (2 ** retries), 900)
                print(f"[Página {page}] Rate limit. Esperando {wait}s...")
                time.sleep(wait)
                retries += 1
                if retries > 5:
                    raise RuntimeError("Demasiados reintentos por rate limit.")
            elif r.status_code == 403:
                raise RuntimeError(
                    f"403 Forbidden: tu plan no tiene acceso a search/all. "
                    f"Necesitas Enterprise, Academic o DSA researcher access.\n"
                    f"Detalle: {r.text}"
                )
            else:
                raise RuntimeError(f"X API error {r.status_code}: {r.text}")

        payload = r.json()
        data = payload.get("data", []) or []
        includes = payload.get("includes", {}) or {}
        meta = payload.get("meta", {}) or {}

        users_by_id = {u["id"]: u for u in includes.get("users", []) or []}
        places_by_id = {p["id"]: p for p in includes.get("places", []) or []}

        for t in data:
            author = users_by_id.get(t.get("author_id"), {})
            place_id = (t.get("geo") or {}).get("place_id")
            place = places_by_id.get(place_id, {}) if place_id else {}

            all_rows.append({
                "tweet_id": t.get("id"),
                "created_at": t.get("created_at"),
                "text": t.get("text"),
                "lang": t.get("lang"),
                "author_id": t.get("author_id"),
                "username": author.get("username"),
                "name": author.get("name"),
                "user_location": author.get("location"),
                "place_id": place_id,
                "place_full_name": place.get("full_name"),
                "place_country": place.get("country"),
                "place_country_code": place.get("country_code"),
                "place_type": place.get("place_type"),
                "retweet_count": (t.get("public_metrics") or {}).get("retweet_count"),
                "reply_count": (t.get("public_metrics") or {}).get("reply_count"),
                "like_count": (t.get("public_metrics") or {}).get("like_count"),
                "quote_count": (t.get("public_metrics") or {}).get("quote_count"),
            })

        print(f"[Página {page + 1}/{max_pages}] Acumulados: {len(all_rows)} tweets")

        next_token = meta.get("next_token")
        if not next_token:
            print("No hay más páginas disponibles.")
            break

        if len(all_rows) >= target_rows:
            print(f"Objetivo de {target_rows} alcanzado.")
            break

        time.sleep(sleep_seconds)

    return pd.DataFrame(all_rows)


In [ ]:
# Carga el token desde una variable de entorno (NUNCA lo pegues hardcoded en el notebook si vas a subirlo a GitHub).
# Crea un archivo .env en la raíz del repo con la línea:
#     X_BEARER_TOKEN=tu_token_aquí
# y carga las variables con python-dotenv (o expórtalas manualmente antes de abrir el notebook).

import os
from dotenv import load_dotenv

load_dotenv()  # busca un .env en el directorio actual o superiores

token = os.getenv("X_BEARER_TOKEN", "")
if not token:
    raise ValueError(
        "Falta X_BEARER_TOKEN. Define la variable en un archivo .env "
        "(ver .env.example) o expórtala en tu shell antes de ejecutar el notebook."
    )

query = (
    "("
    "phishing OR smishing OR vishing "
    "OR \"fake delivery\" OR \"fake invoice\" OR \"USPS scam\" OR \"FedEx scam\" "
    "OR \"customs fee scam\" OR \"toll scam\" OR \"IRS scam\" "
    "OR \"Zelle scam\" OR \"Venmo scam\" OR \"Cash App scam\" OR \"PayPal scam\" "
    "OR \"wire fraud\" OR \"bank scam\" OR \"crypto scam\" OR \"bitcoin scam\" "
    "OR \"rug pull\" OR \"pig butchering\" OR \"investment scam\" OR \"ponzi scheme\" "
    "OR \"romance scam\" OR \"job scam\" OR \"tech support scam\" "
    "OR \"gift card scam\" OR \"check fraud\" OR \"identity theft\" "
    "OR \"got scammed\" OR \"scam text\" OR \"scam email\" "
    "OR ((scam OR fraud) (bank OR card OR account OR wire OR transfer "
    "OR crypto OR bitcoin OR wallet OR loan OR invoice OR refund))"
    ") "
    "-trump -biden -obama -kamala -election -voter -ballot "
    "-\"voter fraud\" -\"election fraud\" -\"government fraud\" "
    "-medicare -medicaid -gop -maga -republican -democrat "
    "-liberal -conservative -woke -nba -nfl -kpop "
    "lang:en place_country:US "
    "-is:retweet -is:reply -is:nullcast"
)

df = search_all_posts_to_df(
        bearer_token=token,
        query=query,
        max_pages=2,                          # 40 × 500 = 20.000 tweets objetivo
        max_results=500,
        sleep_seconds=1.1,
        start_time="2025-01-01T00:00:00Z",    # 1 enero 2025
        end_time="2025-12-31T23:59:59Z",      # 31 diciembre 2025
)

us_keywords = ["united states", "usa", "u.s.", "us ", "america",
               "new york", "california", "texas", "florida", "chicago",
               "los angeles", "boston", "seattle", "miami"]

def looks_us(row):
    if row.get("place_country_code") == "US":
        return True
    loc = str(row.get("user_location") or "").lower()
    return any(k in loc for k in us_keywords)

df["likely_us"] = df.apply(looks_us, axis=1)

# Guardar fuera del repo Git (data/ está en .gitignore para no subir datos brutos a GitHub)
os.makedirs("../data/raw", exist_ok=True)
df.to_csv("../data/raw/scam_us_all_posts_2025.csv", index=False, encoding="utf-8")

print(f"Total filas: {len(df)}")
print(f"Probablemente US: {df['likely_us'].sum()}")
df.head()
